## Transform and Load


##### Main tasks:
##### 1. Load dimensions
##### 2. Generate surrogate keys
##### 3. Implement Patient SCD Type 2
##### 4. Load fact tables
##### 5. Create readmission analysis
##### ============================================================


In [1]:
-- # Imports
-- from pyspark.sql import functions as F
-- from pyspark.sql.window import Window
-- from notebookutils import data
-- from datetime import datetime, timezone

-- # Fabric Warehouse connector
-- import com.microsoft.spark.fabric
-- from com.microsoft.spark.fabric.Constants import Constants

-- # Define source
-- SILVER_LAKEHOUSE = "HP_02_LH_Health_Silver"
-- GOLD_WAREHOUSE = "HP_04_WH_HealthAnalytics"

##### LOAD DimDate

In [2]:
-- ============================================================
-- DimDate load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.encounter
--
-- Target:
--   dim.DimDate
-- ============================================================


-- ============================================================
-- 1. Get Min and Max Encounter Date
-- ============================================================

DECLARE @StartDate DATE;
DECLARE @EndDate   DATE;

SELECT
    @StartDate = MIN(CAST(EncounterDate AS DATE)),
    @EndDate   = MAX(CAST(EncounterDate AS DATE))

FROM [HP_02_LH_Health_Silver].[clinical].[encounter];


-- ============================================================
-- 2. Generate all dates between Min and Max
-- ============================================================

WITH DateSource AS
(
    SELECT
        DATEADD(
            DAY,
            value,
            @StartDate
        ) AS FullDate

    FROM GENERATE_SERIES
    (
        0,
        DATEDIFF(
            DAY,
            @StartDate,
            @EndDate
        )
    )
),

DimDateSource AS
(
    SELECT
        -- Example:
        -- 2026-10-05 -> 20261005
        CAST(
            CONVERT(
                VARCHAR(8),
                FullDate,
                112
            )
            AS INT
        ) AS DateKey,

        FullDate,

        DAY(
            FullDate
        ) AS DayNumber,

        DATENAME(
            WEEKDAY,
            FullDate
        ) AS DayName,

        DATEPART(
            WEEKDAY,
            FullDate
        ) AS DayOfWeekNumber,

        DATEPART(
            DAYOFYEAR,
            FullDate
        ) AS DayOfYear,

        DATEPART(
            ISO_WEEK,
            FullDate
        ) AS WeekNumber,

        MONTH(
            FullDate
        ) AS MonthNumber,

        DATENAME(
            MONTH,
            FullDate
        ) AS MonthName,

        DATEPART(
            QUARTER,
            FullDate
        ) AS QuarterNumber,

        CONCAT(
            'Q',
            DATEPART(
                QUARTER,
                FullDate
            )
        ) AS QuarterName,

        YEAR(
            FullDate
        ) AS YearNumber,

        CONCAT(
            YEAR(FullDate),
            '-',
            RIGHT(
                '0' +
                CAST(
                    MONTH(FullDate)
                    AS VARCHAR(2)
                ),
                2
            )
        ) AS YearMonth,

        CAST(
            CASE
                WHEN DATENAME(
                    WEEKDAY,
                    FullDate
                )
                IN (
                    'Saturday',
                    'Sunday'
                )
                THEN 1
                ELSE 0
            END
            AS BIT
        ) AS IsWeekend

    FROM DateSource
)


-- ============================================================
-- 3. Merge into DimDate
--
-- DateKey is the business/primary matching key.
-- ============================================================

MERGE INTO dim.DimDate AS target

USING DimDateSource AS source

ON target.DateKey = source.DateKey


-- ============================================================
-- Update existing dates
-- ============================================================

WHEN MATCHED THEN

UPDATE SET

    target.FullDate        = source.FullDate,
    target.DayNumber       = source.DayNumber,
    target.DayName         = source.DayName,
    target.DayOfWeekNumber = source.DayOfWeekNumber,
    target.DayOfYear       = source.DayOfYear,
    target.WeekNumber      = source.WeekNumber,
    target.MonthNumber     = source.MonthNumber,
    target.MonthName       = source.MonthName,
    target.QuarterNumber   = source.QuarterNumber,
    target.QuarterName     = source.QuarterName,
    target.YearNumber      = source.YearNumber,
    target.YearMonth       = source.YearMonth,
    target.IsWeekend       = source.IsWeekend


-- ============================================================
-- Insert new dates
-- ============================================================

WHEN NOT MATCHED BY TARGET THEN

INSERT
(
    DateKey,
    FullDate,

    DayNumber,
    DayName,

    DayOfWeekNumber,
    DayOfYear,

    WeekNumber,

    MonthNumber,
    MonthName,

    QuarterNumber,
    QuarterName,

    YearNumber,
    YearMonth,

    IsWeekend
)

VALUES
(
    source.DateKey,
    source.FullDate,

    source.DayNumber,
    source.DayName,

    source.DayOfWeekNumber,
    source.DayOfYear,

    source.WeekNumber,

    source.MonthNumber,
    source.MonthName,

    source.QuarterNumber,
    source.QuarterName,

    source.YearNumber,
    source.YearMonth,

    source.IsWeekend
);


-- ============================================================
-- 4. Verify
-- ============================================================

SELECT
    COUNT(*) AS TotalRows,
    MIN(FullDate) AS MinDate,
    MAX(FullDate) AS MaxDate

FROM dim.DimDate;

#### Load Patient

In [3]:
-- ============================================================
-- Load DimPatient staging table
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.patient
--
-- Target:
--   stg.DimPatient
-- ============================================================


-- ============================================================
-- 1. Clear current staging data
-- Equivalent to .mode("overwrite")
-- ============================================================

TRUNCATE TABLE stg.DimPatient;


-- ============================================================
-- 2. Load staging table from Silver
--
-- Excluded:
--   DeathDate
--   Passport
--   Suffix
--   _ingestion_timestamp
--   _source_system
--   _source_entity
--   _pipeline_run_id
-- ============================================================

INSERT INTO stg.DimPatient
(
    PatientID,
    FirstName,
    LastName,
    BirthDate,
    Gender,
    Race,
    Ethnicity,
    MaritalStatus,
    Age,
    AgeGroup
)

SELECT
    PatientID,
    FirstName,
    LastName,
    BirthDate,
    Gender,
    Race,
    Ethnicity,
    MaritalStatus,
    Age,
    AgeGroup

FROM [HP_02_LH_Health_Silver].[clinical].[patient];


-- ============================================================
-- 3. Verify staging load
-- ============================================================

SELECT
    COUNT(*) AS StagingRows
FROM stg.DimPatient;


SELECT TOP 10 *
FROM stg.DimPatient
ORDER BY PatientID;

In [4]:
-- ============================================================
-- DimPatient SCD Type 2
--
-- Source:
--      stg.DimPatient
--
-- Target:
--      dim.DimPatient
--
-- Business Key:
--      PatientID
--
-- SCD Type 2 attribute:
--      MaritalStatus
--
-- Type 1 attributes:
--      FirstName
--      LastName
--      BirthDate
--      Gender
--      Race
--      Ethnicity
--      Age
--      AgeGroup
-- ============================================================


-- Use one timestamp for the entire SCD load
DECLARE @LoadTime DATETIME2(6) = SYSUTCDATETIME();


-- ============================================================
-- 1. Expire existing current rows when MaritalStatus changes
-- ============================================================

UPDATE target

SET
    target.ValidTo   = @LoadTime,
    target.IsCurrent = 0

FROM dim.DimPatient AS target

INNER JOIN stg.DimPatient AS source
    ON target.PatientID = source.PatientID

WHERE
    target.IsCurrent = 1

    AND ISNULL(target.MaritalStatus, '')
        <> ISNULL(source.MaritalStatus, '');


-- ============================================================
-- 2. Type 1 update
--
-- Only rows still current reach this step.
-- Changed SCD2 rows were expired above.
-- ============================================================

UPDATE target

SET
    target.FirstName = source.FirstName,
    target.LastName  = source.LastName,

    target.BirthDate = source.BirthDate,

    target.Gender    = source.Gender,
    target.Race      = source.Race,
    target.Ethnicity = source.Ethnicity,

    target.Age       = source.Age,
    target.AgeGroup  = source.AgeGroup

FROM dim.DimPatient AS target

INNER JOIN stg.DimPatient AS source
    ON target.PatientID = source.PatientID

WHERE
    target.IsCurrent = 1;


-- ============================================================
-- 3. Get current maximum PatientKey
-- ============================================================

DECLARE @MaxPatientKey BIGINT;

SELECT
    @MaxPatientKey = ISNULL(MAX(PatientKey), 0)

FROM dim.DimPatient;


-- ============================================================
-- 4. Find rows that need inserting
--
-- This automatically includes:
--
--   NEW patient
--       -> no current target row
--
--   CHANGED patient
--       -> old current row was expired in Step 1,
--          therefore there is now no current target row
-- ============================================================

WITH PatientsToInsert AS
(
    SELECT
        source.PatientID,

        source.FirstName,
        source.LastName,

        source.BirthDate,

        source.Gender,
        source.Race,
        source.Ethnicity,

        source.MaritalStatus,

        source.Age,
        source.AgeGroup,


        ROW_NUMBER() OVER
        (
            ORDER BY source.PatientID
        ) AS RowNumber

    FROM stg.DimPatient AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM dim.DimPatient AS target

        WHERE
            target.PatientID = source.PatientID
            AND target.IsCurrent = 1
    )
)


-- ============================================================
-- 5. Insert new patients and new SCD2 versions
-- ============================================================

INSERT INTO dim.DimPatient
(
    PatientKey,
    PatientID,

    FirstName,
    LastName,

    BirthDate,

    Gender,
    Race,
    Ethnicity,

    MaritalStatus,

    Age,
    AgeGroup,


    ValidFrom,
    ValidTo,
    IsCurrent
)

SELECT
    @MaxPatientKey + RowNumber,

    PatientID,

    FirstName,
    LastName,

    BirthDate,

    Gender,
    Race,
    Ethnicity,

    MaritalStatus,

    Age,
    AgeGroup,


    @LoadTime,

    CAST(
        '9999-12-31 23:59:59.999999'
        AS DATETIME2(6)
    ),

    1

FROM PatientsToInsert;


-- ============================================================
-- 6. Summary
-- ============================================================

SELECT
    COUNT(*) AS TotalRows,

    COUNT(DISTINCT PatientID)
        AS UniquePatients,

    SUM(
        CASE
            WHEN IsCurrent = 1
            THEN 1
            ELSE 0
        END
    ) AS CurrentRows,

    SUM(
        CASE
            WHEN IsCurrent = 0
            THEN 1
            ELSE 0
        END
    ) AS HistoricalRows

FROM dim.DimPatient;


-- ============================================================
-- 7. Validate
--
-- Expected result:
--      zero rows
-- ============================================================

SELECT
    PatientID,
    COUNT(*) AS CurrentVersions

FROM dim.DimPatient

WHERE IsCurrent = 1

GROUP BY PatientID

HAVING COUNT(*) > 1;

#### Load DimProcedure

In [5]:
-- ============================================================
-- DimProcedure Load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.procedure
--
-- Target:
--   dim.DimProcedure
--
-- Business Key:
--   ProcedureCode
--
-- Surrogate Key:
--   ProcedureKey
--
-- SCD:
--   Type 1
-- ============================================================


-- ============================================================
-- 1. Prepare one row per ProcedureCode
-- ============================================================

WITH ProcedureRanked AS
(
    SELECT
        ProcedureCode,
        ProcedureDescription,

        ROW_NUMBER() OVER
        (
            PARTITION BY ProcedureCode
            ORDER BY ProcedureDate DESC
        ) AS rn

    FROM [HP_02_LH_Health_Silver].[clinical].[procedure]

    WHERE ProcedureCode IS NOT NULL
),

ProcedureSource AS
(
    SELECT
        ProcedureCode,
        ProcedureDescription

    FROM ProcedureRanked

    WHERE rn = 1
),

MaxKey AS
(
    SELECT
        ISNULL(MAX(ProcedureKey), 0) AS MaxProcedureKey

    FROM dim.DimProcedure
),

ProcedurePrepared AS
(
    SELECT
        s.ProcedureCode,
        s.ProcedureDescription,

        m.MaxProcedureKey
        +
        ROW_NUMBER() OVER
        (
            ORDER BY s.ProcedureCode
        ) AS NewProcedureKey

    FROM ProcedureSource AS s

    CROSS JOIN MaxKey AS m
)


-- ============================================================
-- 2. Merge into DimProcedure
-- ============================================================

MERGE INTO dim.DimProcedure AS target

USING ProcedurePrepared AS source

ON target.ProcedureCode = source.ProcedureCode


-- ============================================================
-- Existing ProcedureCode
-- Update descriptive attributes
-- ============================================================

WHEN MATCHED
AND
(
       ISNULL(target.ProcedureDescription, '')
       <>
       ISNULL(source.ProcedureDescription, '')
)
THEN

UPDATE SET
    target.ProcedureDescription =
        source.ProcedureDescription


-- ============================================================
-- New ProcedureCode
-- Insert new dimension member
-- ============================================================

WHEN NOT MATCHED BY TARGET
THEN

INSERT
(
    ProcedureKey,
    ProcedureCode,
    ProcedureDescription
)

VALUES
(
    source.NewProcedureKey,
    source.ProcedureCode,
    source.ProcedureDescription
);

##### Load DimFacility

In [6]:
-- ============================================================
-- DimFacility - SCD Type 2
--
-- Source:
--   HP_02_LH_Health_Silver.reference.facility
--
-- Target:
--   dim.DimFacility
--
-- Business Key:
--   FacilityID
--
-- Surrogate Key:
--   FacilityKey
--
-- SCD Type 2:
--   FacilityName
--   Address
--   City
--   State
--   ZipCode
--   Latitude
--   Longitude
--   Phone
--
-- Type 1:
--   Revenue
--   Utilization
-- ============================================================


DECLARE @LoadTime DATETIME2(6) = SYSUTCDATETIME();


-- ============================================================
-- 1. Expire current Facility record when an SCD2
--    attribute changes
-- ============================================================

UPDATE target

SET
    target.ValidTo   = @LoadTime,
    target.IsCurrent = 0

FROM dim.DimFacility AS target

INNER JOIN
(
    SELECT
        FacilityID,
        FacilityName,
        Address,
        City,
        State,
        ZipCode,
        Latitude,
        Longitude,
        Phone,
        Revenue,
        Utilization

    FROM [HP_02_LH_Health_Silver].[reference].[facility]

    WHERE FacilityID IS NOT NULL

) AS source

    ON target.FacilityID = source.FacilityID

WHERE
    target.IsCurrent = 1

    AND
    (
           ISNULL(target.FacilityName, '')
           <> ISNULL(source.FacilityName, '')
        AND ISNULL(target.Phone, '')
           <> ISNULL(source.Phone, '')

        OR ISNULL(target.Address, '')
           <> ISNULL(source.Address, '')

        OR ISNULL(target.City, '')
           <> ISNULL(source.City, '')

        AND ISNULL(target.State, '')
           <> ISNULL(source.State, '')

        AND ISNULL(target.ZipCode, '')
           <> ISNULL(source.ZipCode, '')

        AND ISNULL(target.Latitude, 0)
           <> ISNULL(source.Latitude, 0)

        AND ISNULL(target.Longitude, 0)
           <> ISNULL(source.Longitude, 0)

    );


-- ============================================================
-- 2. Find current maximum FacilityKey
-- ============================================================

DECLARE @MaxFacilityKey BIGINT;

SELECT
    @MaxFacilityKey =
        ISNULL(MAX(FacilityKey), 0)

FROM dim.DimFacility;


-- ============================================================
-- 3. Insert:
--
--    A. Completely new facilities
--    B. New versions of facilities expired above
-- ============================================================

WITH FacilitySource AS
(
    SELECT
        FacilityID,
        FacilityName,
        Address,
        City,
        State,
        ZipCode,
        Latitude,
        Longitude,
        Phone,
        Revenue,
        Utilization

    FROM [HP_02_LH_Health_Silver].[reference].[facility]

    WHERE FacilityID IS NOT NULL
),

FacilitiesToInsert AS
(
    SELECT
        source.FacilityID,
        source.FacilityName,
        source.Address,
        source.City,
        source.State,
        source.ZipCode,
        source.Latitude,
        source.Longitude,
        source.Phone,
        source.Revenue,
        source.Utilization,

        ROW_NUMBER() OVER
        (
            ORDER BY source.FacilityID
        ) AS RowNumber

    FROM FacilitySource AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM dim.DimFacility AS target

        WHERE
            target.FacilityID = source.FacilityID
            AND target.IsCurrent = 1
    )
)


INSERT INTO dim.DimFacility
(
    FacilityKey,
    FacilityID,
    FacilityName,

    Address,
    City,
    State,
    ZipCode,

    Latitude,
    Longitude,

    Phone,

    Revenue,
    Utilization,

    ValidFrom,
    ValidTo,
    IsCurrent
)

SELECT
    @MaxFacilityKey + RowNumber,

    FacilityID,
    FacilityName,

    Address,
    City,
    State,
    ZipCode,

    Latitude,
    Longitude,

    Phone,

    Revenue,
    Utilization,

    @LoadTime,

    CAST(
        '9999-12-31 23:59:59.999999'
        AS DATETIME2(6)
    ),

    1

FROM FacilitiesToInsert;


-- ============================================================
-- 4. Type 1 update
--
-- Revenue and Utilization do not create new historical
-- Facility versions.
-- ============================================================

UPDATE target

SET
    target.Revenue     = source.Revenue,
    target.Utilization = source.Utilization

FROM dim.DimFacility AS target

INNER JOIN
(
    SELECT
        FacilityID,
        Revenue,
        Utilization

    FROM [HP_02_LH_Health_Silver].[reference].[facility]

    WHERE FacilityID IS NOT NULL

) AS source

    ON target.FacilityID = source.FacilityID

WHERE
    target.IsCurrent = 1;


-- ============================================================
-- 5. Summary
-- ============================================================

SELECT
    COUNT(*) AS TotalFacilityRows,

    COUNT(DISTINCT FacilityID)
        AS UniqueFacilities,

    SUM(
        CASE
            WHEN IsCurrent = 1
            THEN 1
            ELSE 0
        END
    ) AS CurrentRows,

    SUM(
        CASE
            WHEN IsCurrent = 0
            THEN 1
            ELSE 0
        END
    ) AS HistoricalRows

FROM dim.DimFacility;


-- ============================================================
-- 6. Validate
--
-- Expected:
-- zero rows
-- ============================================================

SELECT
    FacilityID,
    COUNT(*) AS CurrentVersions

FROM dim.DimFacility

WHERE IsCurrent = 1

GROUP BY FacilityID

HAVING COUNT(*) > 1;

##### Load DimProvider

In [7]:
-- ============================================================
-- DimProvider - SCD Type 2
--
-- Source:
--   HP_02_LH_Health_Silver.reference.provider
--
-- Target:
--   dim.DimProvider
--
-- Business Key:
--   ProviderID
--
-- Surrogate Key:
--   ProviderKey
--
-- SCD Type 2:
--   FacilityID
--   Address
--   City
--   State
--   ZipCode
--   Latitude
--   Longitude
--
-- Type 1:
--   ProviderName
--   Gender
--   Specialty
-- ============================================================


DECLARE @LoadTime DATETIME2(6) = SYSUTCDATETIME();


-- ============================================================
-- 1. Expire current provider record when an SCD2
--    attribute changes
-- ============================================================

UPDATE target

SET
    target.ValidTo   = @LoadTime,
    target.IsCurrent = 0

FROM dim.DimProvider AS target

INNER JOIN
(
    SELECT
        ProviderID,
        FacilityID,
        ProviderName,
        Gender,
        Specialty,
        Address,
        City,
        State,
        ZipCode,
        Latitude,
        Longitude

    FROM [HP_02_LH_Health_Silver].[reference].[provider]

    WHERE ProviderID IS NOT NULL

) AS source

    ON target.ProviderID = source.ProviderID

WHERE
    target.IsCurrent = 1

    AND
    (
           ISNULL(target.FacilityID, '')
           <> ISNULL(source.FacilityID, '')

        OR ISNULL(target.Address, '')
           <> ISNULL(source.Address, '')

        OR ISNULL(target.City, '')
           <> ISNULL(source.City, '')

        OR ISNULL(target.State, '')
           <> ISNULL(source.State, '')

        OR ISNULL(target.ZipCode, '')
           <> ISNULL(source.ZipCode, '')

        OR ISNULL(target.Latitude, 0)
           <> ISNULL(source.Latitude, 0)

        OR ISNULL(target.Longitude, 0)
           <> ISNULL(source.Longitude, 0)
    );


-- ============================================================
-- 2. Find current maximum ProviderKey
-- ============================================================

DECLARE @MaxProviderKey BIGINT;

SELECT
    @MaxProviderKey =
        ISNULL(MAX(ProviderKey), 0)

FROM dim.DimProvider;


-- ============================================================
-- 3. Prepare source
-- ============================================================

WITH ProviderSource AS
(
    SELECT
        ProviderID,
        FacilityID,
        ProviderName,
        Gender,
        Specialty,
        Address,
        City,
        State,
        ZipCode,
        Latitude,
        Longitude

    FROM [HP_02_LH_Health_Silver].[reference].[provider]

    WHERE ProviderID IS NOT NULL
),


-- ============================================================
-- 4. Find providers that need inserting
--
-- Includes:
--   1. Completely new providers
--   2. New versions of providers expired in Step 1
-- ============================================================

ProvidersToInsert AS
(
    SELECT
        source.ProviderID,
        source.FacilityID,
        source.ProviderName,
        source.Gender,
        source.Specialty,
        source.Address,
        source.City,
        source.State,
        source.ZipCode,
        source.Latitude,
        source.Longitude,

        ROW_NUMBER() OVER
        (
            ORDER BY source.ProviderID
        ) AS RowNumber

    FROM ProviderSource AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM dim.DimProvider AS target

        WHERE
            target.ProviderID = source.ProviderID
            AND target.IsCurrent = 1
    )
)


-- ============================================================
-- 5. Insert new providers and new SCD2 versions
-- ============================================================

INSERT INTO dim.DimProvider
(
    ProviderKey,
    ProviderID,

    FacilityID,

    ProviderName,
    Gender,
    Specialty,

    Address,
    City,
    State,
    ZipCode,

    Latitude,
    Longitude,

    ValidFrom,
    ValidTo,
    IsCurrent
)

SELECT
    @MaxProviderKey + RowNumber,

    ProviderID,

    FacilityID,

    ProviderName,
    Gender,
    Specialty,

    Address,
    City,
    State,
    ZipCode,

    Latitude,
    Longitude,

    @LoadTime,

    CAST(
        '9999-12-31 23:59:59.999999'
        AS DATETIME2(6)
    ),

    1

FROM ProvidersToInsert;


-- ============================================================
-- 6. Type 1 update
--
-- ProviderName, Gender and Specialty are updated on the
-- current record without creating another historical version.
-- ============================================================

UPDATE target

SET
    target.ProviderName = source.ProviderName,
    target.Gender       = source.Gender,
    target.Specialty    = source.Specialty

FROM dim.DimProvider AS target

INNER JOIN
(
    SELECT
        ProviderID,
        ProviderName,
        Gender,
        Specialty

    FROM [HP_02_LH_Health_Silver].[reference].[provider]

    WHERE ProviderID IS NOT NULL

) AS source

    ON target.ProviderID = source.ProviderID

WHERE
    target.IsCurrent = 1;


-- ============================================================
-- 7. Summary
-- ============================================================

SELECT
    COUNT(*) AS TotalProviderRows,

    COUNT(DISTINCT ProviderID)
        AS UniqueProviders,

    SUM(
        CASE
            WHEN IsCurrent = 1
            THEN 1
            ELSE 0
        END
    ) AS CurrentRows,

    SUM(
        CASE
            WHEN IsCurrent = 0
            THEN 1
            ELSE 0
        END
    ) AS HistoricalRows

FROM dim.DimProvider;


-- ============================================================
-- 8. Validate
--
-- Expected result:
--   zero rows
-- ============================================================

SELECT
    ProviderID,
    COUNT(*) AS CurrentVersions

FROM dim.DimProvider

WHERE IsCurrent = 1

GROUP BY ProviderID

HAVING COUNT(*) > 1;

##### Load DimPayer

In [8]:
-- ============================================================
-- DimPayer - SCD Type 2
--
-- Source:
--   HP_02_LH_Health_Silver.reference.payer
--
-- Target:
--   dim.DimPayer
--
-- Business Key:
--   PayerID
--
-- Surrogate Key:
--   PayerKey
--
-- SCD Type 2 attributes:
--   PayerName
--   Ownership
--   Address
--   City
--   State
--   ZipCode
--   Phone
--
-- State is derived from ZipCode using DimFacility.
-- ============================================================


DECLARE @LoadTime DATETIME2(6) = SYSUTCDATETIME();


-- ============================================================
-- 1. Expire current payer version when any SCD2
--    attribute changes
-- ============================================================

WITH FacilityZipState AS
(
    SELECT
        LEFT(
            LTRIM(RTRIM(CAST(ZipCode AS VARCHAR(20)))),
            5
        ) AS ZipCode,

        MAX(State) AS State

    FROM dim.DimFacility

    WHERE
        IsCurrent = 1
        AND ZipCode IS NOT NULL
        AND State IS NOT NULL

    GROUP BY
        LEFT(
            LTRIM(RTRIM(CAST(ZipCode AS VARCHAR(20)))),
            5
        )

    HAVING COUNT(DISTINCT State) = 1
),

PayerSource AS
(
    SELECT
        p.PayerID,
        p.PayerName,
        p.Ownership,

        p.Address,
        p.City,

        -- Derive State from ZIP.
        -- Fall back to source StateHeadquartered
        -- when ZIP is not found in DimFacility.
        COALESCE(
            f.State,
            p.StateHeadquartered
        ) AS State,

        p.ZipCode,
        p.Phone

    FROM
        [HP_02_LH_Health_Silver].[reference].[payer] AS p

    LEFT JOIN FacilityZipState AS f

        ON LEFT(
            LTRIM(RTRIM(CAST(p.ZipCode AS VARCHAR(20)))),
            5
        ) = f.ZipCode

    WHERE
        p.PayerID IS NOT NULL
)

UPDATE target

SET
    target.ValidTo   = @LoadTime,
    target.IsCurrent = 0

FROM dim.DimPayer AS target

INNER JOIN PayerSource AS source
    ON target.PayerID = source.PayerID

WHERE
    target.IsCurrent = 1

    AND
    (
           ISNULL(target.PayerName, '')
           <> ISNULL(source.PayerName, '')

        OR ISNULL(target.Ownership, '')
           <> ISNULL(source.Ownership, '')

        OR ISNULL(target.Address, '')
           <> ISNULL(source.Address, '')

        OR ISNULL(target.City, '')
           <> ISNULL(source.City, '')

        OR ISNULL(target.State, '')
           <> ISNULL(source.State, '')

        OR ISNULL(target.ZipCode, '')
           <> ISNULL(source.ZipCode, '')

        OR ISNULL(target.Phone, '')
           <> ISNULL(source.Phone, '')
    );


-- ============================================================
-- 2. Find current maximum PayerKey
-- ============================================================

DECLARE @MaxPayerKey BIGINT;

SELECT
    @MaxPayerKey =
        ISNULL(MAX(PayerKey), 0)

FROM dim.DimPayer;


-- ============================================================
-- 3. Prepare source again for INSERT
-- ============================================================

WITH FacilityZipState AS
(
    SELECT
        LEFT(
            LTRIM(RTRIM(CAST(ZipCode AS VARCHAR(20)))),
            5
        ) AS ZipCode,

        MAX(State) AS State

    FROM dim.DimFacility

    WHERE
        IsCurrent = 1
        AND ZipCode IS NOT NULL
        AND State IS NOT NULL

    GROUP BY
        LEFT(
            LTRIM(RTRIM(CAST(ZipCode AS VARCHAR(20)))),
            5
        )

    HAVING COUNT(DISTINCT State) = 1
),

PayerSource AS
(
    SELECT
        p.PayerID,
        p.PayerName,
        p.Ownership,

        p.Address,
        p.City,

        COALESCE(
            f.State,
            p.StateHeadquartered
        ) AS State,

        p.ZipCode,
        p.Phone

    FROM
        [HP_02_LH_Health_Silver].[reference].[payer] AS p

    LEFT JOIN FacilityZipState AS f

        ON LEFT(
            LTRIM(RTRIM(CAST(p.ZipCode AS VARCHAR(20)))),
            5
        ) = f.ZipCode

    WHERE
        p.PayerID IS NOT NULL
),

PayersToInsert AS
(
    SELECT
        source.PayerID,
        source.PayerName,
        source.Ownership,

        source.Address,
        source.City,
        source.State,
        source.ZipCode,
        source.Phone,

        ROW_NUMBER() OVER
        (
            ORDER BY source.PayerID
        ) AS RowNumber

    FROM PayerSource AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM dim.DimPayer AS target

        WHERE
            target.PayerID = source.PayerID
            AND target.IsCurrent = 1
    )
)


-- ============================================================
-- 4. Insert new payer or new SCD2 version
-- ============================================================

INSERT INTO dim.DimPayer
(
    PayerKey,
    PayerID,
    PayerName,
    Ownership,

    Address,
    City,
    State,
    ZipCode,
    Phone,

    ValidFrom,
    ValidTo,
    IsCurrent
)

SELECT
    @MaxPayerKey + RowNumber,

    PayerID,
    PayerName,
    Ownership,

    Address,
    City,
    State,
    ZipCode,
    Phone,

    @LoadTime,

    CAST(
        '9999-12-31 23:59:59.999999'
        AS DATETIME2(6)
    ),

    1

FROM PayersToInsert;


-- ============================================================
-- 5. Summary
-- ============================================================

SELECT
    COUNT(*) AS TotalPayerRows,

    COUNT(DISTINCT PayerID)
        AS UniquePayers,

    SUM(
        CASE
            WHEN IsCurrent = 1
            THEN 1
            ELSE 0
        END
    ) AS CurrentRows,

    SUM(
        CASE
            WHEN IsCurrent = 0
            THEN 1
            ELSE 0
        END
    ) AS HistoricalRows

FROM dim.DimPayer;


-- ============================================================
-- 6. Validate
--
-- Should return zero rows.
-- ============================================================

SELECT
    PayerID,
    COUNT(*) AS CurrentVersions

FROM dim.DimPayer

WHERE IsCurrent = 1

GROUP BY PayerID

HAVING COUNT(*) > 1;

##### Load DimMedication

In [9]:
-- ============================================================
-- DimMedication Load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.medication
--
-- Target:
--   dim.DimMedication
--
-- Business Key:
--   MedicationCode
--
-- Surrogate Key:
--   MedicationKey
--
-- SCD:
--   Type 1
-- ============================================================


-- ============================================================
-- 1. Find current maximum MedicationKey
-- ============================================================

DECLARE @MaxMedicationKey BIGINT;

SELECT
    @MaxMedicationKey = ISNULL(MAX(MedicationKey), 0)
FROM dim.DimMedication;


-- ============================================================
-- 2. MERGE MedicationSource into DimMedication
--
-- Existing MedicationCode:
--   Update MedicationDescription
--
-- New MedicationCode:
--   Insert new dimension member
-- ============================================================

WITH MedicationSource AS
(
    SELECT
        MedicationCode,

        MAX(MedicationDescription) AS MedicationDescription

    FROM [HP_02_LH_Health_Silver].[clinical].[medication]

    WHERE MedicationCode IS NOT NULL

    GROUP BY
        MedicationCode
),

NewMedicationKeys AS
(
    SELECT
        MedicationCode,

        @MaxMedicationKey
        + ROW_NUMBER() OVER
        (
            ORDER BY MedicationCode
        ) AS MedicationKey

    FROM MedicationSource

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM dim.DimMedication AS target

        WHERE target.MedicationCode =
              MedicationSource.MedicationCode
    )
)

MERGE dim.DimMedication AS target

USING
(
    SELECT
        source.MedicationCode,
        source.MedicationDescription,
        newKeys.MedicationKey

    FROM MedicationSource AS source

    LEFT JOIN NewMedicationKeys AS newKeys
        ON newKeys.MedicationCode =
           source.MedicationCode

) AS source

ON target.MedicationCode =
   source.MedicationCode


-- ============================================================
-- Existing MedicationCode
-- SCD Type 1: overwrite description
-- ============================================================

WHEN MATCHED
     AND ISNULL(target.MedicationDescription, '')
         <>
         ISNULL(source.MedicationDescription, '')

THEN
    UPDATE SET
        target.MedicationDescription =
            source.MedicationDescription


-- ============================================================
-- New MedicationCode
-- Insert new dimension member
-- ============================================================

WHEN NOT MATCHED BY TARGET

THEN
    INSERT
    (
        MedicationKey,
        MedicationCode,
        MedicationDescription
    )

    VALUES
    (
        source.MedicationKey,
        source.MedicationCode,
        source.MedicationDescription
    );


-- ============================================================
-- 3. Summary
-- ============================================================

SELECT
    COUNT(*) AS TotalMedicationRows,

    COUNT(DISTINCT MedicationCode)
        AS UniqueMedicationCodes

FROM dim.DimMedication;


-- ============================================================
-- 4. Validate duplicate MedicationCodes
--
-- Expected:
--   zero rows
-- ============================================================

SELECT
    MedicationCode,
    COUNT(*) AS RowCounts

FROM dim.DimMedication

GROUP BY
    MedicationCode

HAVING COUNT(*) > 1;


##### Load DimDiagnosis


In [10]:
-- ============================================================
-- DimDiagnosis Load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.condition
--
-- Target:
--   dim.DimDiagnosis
--
-- Business Key:
--   DiagnosisCode = ConditionCode
--
-- Surrogate Key:
--   DiagnosisKey
--
-- SCD:
--   Type 1
-- ============================================================


-- ============================================================
-- 1. Get current maximum DiagnosisKey
-- ============================================================

DECLARE @MaxDiagnosisKey BIGINT;

SELECT
    @MaxDiagnosisKey =
        ISNULL(MAX(DiagnosisKey), 0)

FROM dim.DimDiagnosis;


-- ============================================================
-- 2. Update description when DiagnosisCode already exists
-- ============================================================

UPDATE target

SET
    target.DiagnosisDescription =
        source.DiagnosisDescription

FROM dim.DimDiagnosis AS target

INNER JOIN
(
    SELECT
        ConditionCode AS DiagnosisCode,

        MAX(ConditionDescription)
            AS DiagnosisDescription

    FROM [HP_02_LH_Health_Silver].[clinical].[condition]

    WHERE ConditionCode IS NOT NULL

    GROUP BY ConditionCode

) AS source

    ON target.DiagnosisCode =
       source.DiagnosisCode

WHERE
    ISNULL(target.DiagnosisDescription, '')
    <>
    ISNULL(source.DiagnosisDescription, '');


-- ============================================================
-- 3. Prepare unique diagnosis codes
-- ============================================================

WITH DiagnosisSource AS
(
    SELECT
        ConditionCode AS DiagnosisCode,

        MAX(ConditionDescription)
            AS DiagnosisDescription

    FROM [HP_02_LH_Health_Silver].[clinical].[condition]

    WHERE ConditionCode IS NOT NULL

    GROUP BY ConditionCode
),


-- ============================================================
-- 4. Find only NEW DiagnosisCodes
-- ============================================================

NewDiagnoses AS
(
    SELECT
        source.DiagnosisCode,
        source.DiagnosisDescription,

        ROW_NUMBER() OVER
        (
            ORDER BY source.DiagnosisCode
        ) AS RowNumber

    FROM DiagnosisSource AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM dim.DimDiagnosis AS target

        WHERE
            target.DiagnosisCode =
            source.DiagnosisCode
    )
)


-- ============================================================
-- 5. Insert new diagnosis dimension members
-- ============================================================

INSERT INTO dim.DimDiagnosis
(
    DiagnosisKey,
    DiagnosisCode,
    DiagnosisDescription
)

SELECT
    @MaxDiagnosisKey + RowNumber,

    DiagnosisCode,
    DiagnosisDescription

FROM NewDiagnoses;


-- ============================================================
-- 6. Summary
-- ============================================================

SELECT
    COUNT(*) AS TotalDiagnosisRows,

    COUNT(DISTINCT DiagnosisCode)
        AS UniqueDiagnosisCodes

FROM dim.DimDiagnosis;


-- ============================================================
-- 7. Validate
--
-- Expected:
-- zero rows
-- ============================================================

SELECT
    DiagnosisCode,
    COUNT(*) AS RecentChanges

FROM dim.DimDiagnosis

GROUP BY DiagnosisCode

HAVING COUNT(*) > 1;